# 🤖 Notebook 03 — Modelling, Evaluation & SHAP
## MarocChurn · Telecom Customer Churn Prediction
**Author:** Ahmed Mansof · Master Data Science · ENS Tétouan

---

### 🎯 Objectives
1. Train 5 ML models and compare performance
2. Evaluate with Accuracy, F1, ROC-AUC, Confusion Matrix
3. Tune the best model with GridSearchCV
4. Explain predictions globally and locally with SHAP
5. Save the final model

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, time, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay)
from sklearn.model_selection import GridSearchCV, cross_val_score
import xgboost as xgb
import shap

plt.rcParams.update({
    'figure.facecolor':'#0d1e2d','axes.facecolor':'#0d1e2d','axes.edgecolor':'#1a3348',
    'axes.labelcolor':'white','xtick.color':'white','ytick.color':'white','text.color':'white',
    'grid.color':'#1a3348','grid.alpha':0.5,
})
CYAN='#00e5ff'; PURPLE='#7b61ff'; ORANGE='#ff6b35'; MUTED='#6b8fa3'
print("✅ Imports OK")

## 1. Load Preprocessed Data

In [ ]:
# ── Load processed data ──────────────────────────────────────
try:
    X_train = pd.read_csv('../data/processed/X_train.csv')
    X_test  = pd.read_csv('../data/processed/X_test.csv')
    y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
    y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()
    print("✅ Loaded from data/processed/")
except FileNotFoundError:
    print("⚠️  Processed files not found. Running preprocessing inline...")
    # (re-run inline if needed — see notebook 02)
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder, MinMaxScaler
    from imblearn.over_sampling import SMOTE
    import sys; sys.path.insert(0,'..')
    exec(open('../src/preprocess.py').read()) if os.path.exists('../src/preprocess.py') else None

print(f"X_train: {X_train.shape} | Churn: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"X_test:  {X_test.shape}  | Churn: {y_test.sum()}  ({y_test.mean()*100:.1f}%)")
print(f"\nFeatures: {list(X_train.columns)}")

## 2. Train & Compare 5 Models

In [ ]:
# ── Define models ────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=200, max_depth=5,
                               learning_rate=0.1, subsample=0.9,
                               colsample_bytree=0.8, use_label_encoder=False,
                               eval_metric='logloss', random_state=42),
}

# ── Train and evaluate ────────────────────────────────────────
results = {}
trained_models = {}

print(f"{'Model':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>9} {'Time':>7}")
print("-" * 78)

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
        'Time(s)':   round(elapsed, 2),
    }
    trained_models[name] = model

    r = results[name]
    print(f"{name:<25} {r['Accuracy']:>9.4f} {r['Precision']:>10.4f} "
          f"{r['Recall']:>8.4f} {r['F1-Score']:>8.4f} {r['ROC-AUC']:>9.4f} {elapsed:>6.1f}s")

In [ ]:
# ── Results DataFrame ────────────────────────────────────────
df_results = pd.DataFrame(results).T.sort_values('ROC-AUC', ascending=False)
df_results.index.name = 'Model'
print("\n📊 Results sorted by ROC-AUC:")
display((df_results * 100).round(2).rename(columns={
    'Accuracy':'Accuracy %','Precision':'Precision %','Recall':'Recall %',
    'F1-Score':'F1 %','ROC-AUC':'ROC-AUC %'}).drop(columns=['Time(s)']))

## 3. Visual Comparison

In [ ]:
# ── Visual comparison ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
metrics_plot = ['Accuracy', 'F1-Score', 'ROC-AUC']
colors_bar   = [CYAN, PURPLE, ORANGE]
x = np.arange(len(results))
width = 0.25

for i, (metric, color) in enumerate(zip(metrics_plot, colors_bar)):
    vals = [results[m][metric] for m in results]
    axes[0].bar(x + i*width, vals, width, label=metric, color=color, alpha=0.85)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(list(results.keys()), rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Score')
axes[0].set_title('Models Comparison — All Metrics')
axes[0].legend(facecolor='#0d1e2d', edgecolor='#1a3348')
axes[0].set_ylim(0.6, 1.0)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

# ROC curves
palette = [CYAN, PURPLE, ORANGE, '#00e676', '#ff4444']
for (name, model), color in zip(trained_models.items(), palette):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1], 'w--', lw=1, alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All Models')
axes[1].legend(facecolor='#0d1e2d', edgecolor='#1a3348', fontsize=8)

import matplotlib.ticker as mtick
plt.tight_layout()
plt.savefig('../data/processed/plot_model_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

## 4. Best Model — Detailed Evaluation

In [ ]:
# ── XGBoost — Detailed evaluation ────────────────────────────
best = trained_models['XGBoost']
y_pred = best.predict(X_test)
y_prob = best.predict_proba(X_test)[:, 1]

print("=" * 55)
print(" 🏆 XGBOOST — DETAILED RESULTS")
print("=" * 55)
print(classification_report(y_test, y_pred,
      target_names=['No Churn', 'Churn']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('XGBoost — Confusion Matrix & Feature Importance', fontsize=13)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'],
            linewidths=1, linecolor='#060d16',
            annot_kws={'size':14,'color':'white'})
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# Feature importance
fi = pd.Series(best.feature_importances_, index=X_train.columns)
fi = fi.sort_values(ascending=True).tail(15)
colors_fi = [ORANGE if v > fi.quantile(0.8) else (CYAN if v > fi.quantile(0.5) else PURPLE)
             for v in fi.values]
axes[1].barh(fi.index, fi.values, color=colors_fi, edgecolor='none')
axes[1].set_xlabel('Feature Importance Score')
axes[1].set_title('Top 15 Feature Importances')
for i, (feat, val) in enumerate(zip(fi.index, fi.values)):
    axes[1].text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/plot_xgb_evaluation.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

## 5. Hyperparameter Tuning with GridSearchCV

In [ ]:
# ── GridSearchCV ─────────────────────────────────────────────
print("⚙️  Running GridSearchCV (this may take 2-5 minutes)...")

param_grid = {
    'n_estimators':    [100, 200],
    'max_depth':       [4, 5, 6],
    'learning_rate':   [0.05, 0.1],
    'subsample':       [0.8, 0.9],
    'colsample_bytree':[0.8, 1.0],
}

xgb_cv = xgb.XGBClassifier(use_label_encoder=False,
                             eval_metric='logloss', random_state=42)
grid_search = GridSearchCV(
    xgb_cv, param_grid,
    cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=0
)

t0 = time.time()
grid_search.fit(X_train, y_train)
print(f"✅ Done in {time.time()-t0:.1f}s")

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

# Evaluate best
best_tuned = grid_search.best_estimator_
y_pred_tuned = best_tuned.predict(X_test)
y_prob_tuned = best_tuned.predict_proba(X_test)[:, 1]
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"Test F1-Score: {f1_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"Test ROC-AUC:  {roc_auc_score(y_test, y_prob_tuned):.4f}")

## 6. SHAP — Global & Local Explainability

In [ ]:
# ── SHAP values ──────────────────────────────────────────────
print("⚙️  Computing SHAP values...")
explainer  = shap.TreeExplainer(best_tuned)
X_shap     = X_test.sample(min(300, len(X_test)), random_state=42)
shap_vals  = explainer.shap_values(X_shap)
print("✅ SHAP values computed")

# Summary plot (global importance)
fig1, ax1 = plt.subplots(figsize=(10, 6))
fig1.patch.set_facecolor('#0d1e2d')
shap.summary_plot(shap_vals, X_shap, show=False, plot_type='bar',
                  color=CYAN, plot_size=None)
plt.title('SHAP — Global Feature Importance (Bar)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/plot_shap_summary_bar.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

In [ ]:
# ── SHAP Beeswarm ────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 7))
fig2.patch.set_facecolor('#0d1e2d')
shap.summary_plot(shap_vals, X_shap, show=False, plot_size=None)
plt.title('SHAP Beeswarm — Feature Impact & Direction', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/plot_shap_beeswarm.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

In [ ]:
# ── SHAP Individual explanation ──────────────────────────────
# Explain a HIGH RISK customer (pick one with churn prediction)
y_prob_shap = best_tuned.predict_proba(X_shap)[:, 1]
high_risk_idx = np.argmax(y_prob_shap)  # customer with highest churn prob

customer = X_shap.iloc[[high_risk_idx]]
shap_row  = explainer.shap_values(customer)
prob      = y_prob_shap[high_risk_idx]

print(f"🔴 High-risk customer (index {high_risk_idx})")
print(f"   Churn probability: {prob*100:.1f}%")
print(f"   Prediction: {'CHURN' if prob > 0.5 else 'STAY'}")

fig3, ax3 = plt.subplots(figsize=(10, 5))
fig3.patch.set_facecolor('#0d1e2d')
shap.waterfall_plot(
    shap.Explanation(
        values=shap_row[0],
        base_values=explainer.expected_value,
        data=customer.values[0],
        feature_names=list(X_train.columns)
    ),
    show=False
)
plt.title(f'SHAP Waterfall — Why this customer churns (P={prob*100:.1f}%)',
          color='white', fontsize=11, pad=12)
plt.tight_layout()
plt.savefig('../data/processed/plot_shap_waterfall.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

## 7. Save Final Model

In [ ]:
# ── Save model ───────────────────────────────────────────────
os.makedirs('../models', exist_ok=True)

with open('../models/best_model_xgb.pkl', 'wb') as f:
    pickle.dump(best_tuned, f)

# Save all models for comparison
with open('../models/all_models.pkl', 'wb') as f:
    pickle.dump(trained_models, f)

print("✅ Models saved:")
print("   models/best_model_xgb.pkl")
print("   models/all_models.pkl")

# Final summary
print("\n" + "=" * 60)
print(" 🏆 FINAL RESULTS — XGBoost (tuned)")
print("=" * 60)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"  Precision : {precision_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"  Recall    : {recall_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"  F1-Score  : {f1_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_tuned):.4f}")
print("=" * 60)
print("\n✅ Modelling complete. Run: streamlit run app.py")